# LifeMate: predict notebook (timing)

Same code path as `/code/predict.sh`: both call the functions of `/code/predict.py`, with the same config
(`configs/docker-final.yaml`), the same models, the same environment variables and the same seeds, so the predictions
in `/result/jupyter_submission.jsonl` are those of `/result/submission.jsonl`; only `time` is measured again.
Run all cells in order (*Run → Run All Cells*). Outputs of this notebook use the prefix `jupyter_`:
`/result/jupyter_submission.jsonl` and logs in `/result/jupyter_logs/`.

Pipeline: PhoWhisper-large (base + 2 LoRA adapters, two passes) → Qwen3.5-4B agent (vLLM, temperature 0) → ReAct JSON
tool calls through the split's own `zaic_tools.ToolEnv` (one `ToolEnv` per test case).
`time` of a test case (ms) = its ASR time (its share of the batched ASR, both passes, model loading excluded)
+ its agent time (from the first LLM call to the final state). Model loading is reported separately.

## Step 1: fixed seed

In [ ]:
import os, sys, json, time
from pathlib import Path

sys.path.insert(0, "/code")
import predict as P  # the driver used by /code/predict.sh

SEED = P.SEED  # 0: vLLM --seed 0, request seed 0, temperature 0, PYTHONHASHSEED=0 in every pipeline process
P.seed_everything(SEED)
print("seed", SEED)

## Step 2: resources and test cases

Finds the split under `/data` (`private_test` when present), reads every test case, prepares the offline environment (no network, no HuggingFace download: every model is inside the image).

In [ ]:
ROOT = Path(os.environ.get("DATA", "/data"))
run = P.Run(work_name="jupyter_logs", submission="jupyter_submission.jsonl", log_name="jupyter_predict.log")
test_cases = run.prepare()  # list of dicts from /data/<split>/<split>.jsonl, file order
SPLIT = run.split
print(f"split {SPLIT}: {len(test_cases)} test cases; zaic_tools of the split: {run.data / SPLIT / 'zaic_tools'}")
errors = []

## Step 3: speech recognition (loads PhoWhisper once, transcribes every test case)

Batched exactly as in `predict.sh` (32 test cases per chunk); each test case keeps its own ASR seconds for `time`.

In [ ]:
try:
    run.run_asr()
except RuntimeError as e:
    errors.append(str(e)); run.log(f"ERROR: {e}")

## Step 4: load the LLM (vLLM server on 127.0.0.1, Qwen3.5-4B merged agent)

In [ ]:
if not errors:
    try:
        run.start_llm()
    except RuntimeError as e:
        errors.append(str(e)); run.log(f"ERROR: {e}")

## Step 5: predict every test case and measure its time

One `ToolEnv(SPLIT, id, root=/data)` per test case; the agent runs the test cases one after another.

In [ ]:
agent_run = None
if not errors:
    try:
        agent_run = run.run_agent()
    except RuntimeError as e:
        errors.append(str(e)); run.log(f"ERROR: {e}")
run.stop_llm()
all_result = run.results(agent_run)  # [{id, tool_calls, state_hash, time}] in test-case order
for r in all_result[:5]:
    print(r["id"], r["time"], "ms", [c["name"] for c in r["tool_calls"]])

## Step 6: write `/result/jupyter_submission.jsonl`

In [ ]:
path = run.write(all_result)
print(path, len(all_result), "lines")
run.summary(all_result)
run.close()
if errors:
    print("WARNING: degraded output:", errors)